# Step 3. 비지도학습 맞춤형 정밀 EDA 및 전처리
> K-Means 군집화 거리 왜곡을 원천 방지하기 위한 1% 및 99% 분위수 클리핑 전처리 기법 적용


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

BASE_DIR = Path(".")
DATA_RAW = BASE_DIR / "data" / "raw"
RESULTS_FIGURES = BASE_DIR / "results" / "figures"

plt.rc('font', family='sans-serif')
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

def advanced_anomaly_eda_and_clean(file_path, name):
    df = pd.read_csv(file_path)
    df["date"] = pd.to_datetime(df["date"])
    
    # 1. 무한대 값 및 결측치 안전 정제 (연산 장애 차단)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df["volume"] = df["volume"].replace(0, np.nan)
    df.dropna(subset=["open", "high", "low", "close", "volume"], inplace=True)
    
    # 2. K-Means 거리 왜곡 방지를 위한 금융 표준 1% 및 99% 분위수 클리핑
    for col in ["close", "volume"]:
        lower_bound = df[col].quantile(0.01)
        upper_bound = df[col].quantile(0.99)
        df[col] = df[col].clip(lower_bound, upper_bound)
        
    df.to_csv(file_path, index=False, encoding="utf-8-sig")
    print(f"📊 [{name}] 비지도학습 최적화 정밀 EDA 및 분위수 스케일 클리핑 완료")
    return df

train_eda = advanced_anomaly_eda_and_clean(DATA_RAW / "train.csv", "Train Set")
valid_eda = advanced_anomaly_eda_and_clean(DATA_RAW / "valid.csv", "Valid Set")
test_eda = advanced_anomaly_eda_and_clean(DATA_RAW / "test.csv", "Test Set")
